In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import classification_report, cohen_kappa_score, accuracy_score, mean_absolute_error
import os

PROCESSED_DIR = '../data/processed/'
df_train = pd.read_parquet(PROCESSED_DIR + 'train_cleaned.parquet')
df_valid = pd.read_parquet(PROCESSED_DIR + 'valid_cleaned.parquet')

df_train['label'] = df_train['label'].astype(np.int64)
df_valid['label'] = df_valid['label'].astype(np.int64)

datasets = DatasetDict({
    'train': Dataset.from_pandas(df_train[['Sentence_Normalized', 'label']]),
    'valid': Dataset.from_pandas(df_valid[['Sentence_Normalized', 'label']])
})
print(f"Dữ liệu sẵn sàng! Train: {len(df_train)} câu | Valid: {len(df_valid)} câu")

Dữ liệu sẵn sàng! Train: 54626 câu | Valid: 7310 câu


In [2]:
MODEL_NAME = "UBC-NLP/ARBERTv2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["Sentence_Normalized"], 
        padding="max_length", 
        truncation=True, 
        max_length=128  
    )

print("Đang Tokenize bằng ArBERTv2...")
tokenized_datasets = datasets.map(tokenize_function, batched=True)
tokenized_datasets.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
print("Hoàn tất Tokenize!")

tokenizer_config.json:   0%|          | 0.00/485 [00:00<?, ?B/s]

d:\BAREC-HP-2026\env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ADMIN\.cache\huggingface\hub\models--UBC-NLP--ARBERTv2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Đang Tokenize bằng ArBERTv2...


Map:   0%|          | 0/54626 [00:00<?, ? examples/s]

Map:   0%|          | 0/7310 [00:00<?, ? examples/s]

Hoàn tất Tokenize!


In [3]:
from sklearn.utils.class_weight import compute_class_weight
import torch.nn.functional as F

print("TÍNH TOÁN TRỌNG SỐ MẪU (SAMPLE-LEVEL WEIGHTS)...")
# 1. Calculate balanced class weights
y_train = df_train['label'].values
cw = compute_class_weight('balanced', classes=np.arange(19), y=y_train)

# 2. Clip class weights
cw_clipped = np.clip(cw, 0.5, 5.0)

# 3. Normalize class weights
cw_normalized = cw_clipped / cw_clipped.mean()
class_weights_tensor = torch.tensor(cw_normalized, dtype=torch.float32)

print("Class weights (đã clip và chuẩn hóa):")
print(np.round(cw_normalized, 2))

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=18
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["query", "value", "dense"] 
)
model = get_peft_model(model, lora_config)

class CORALTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        device = logits.device
        
        target = torch.zeros_like(logits, device=device)
        for k in range(logits.shape[1]):
            target[:, k] = (labels > k).float()
            
        loss_per_sample = F.binary_cross_entropy_with_logits(
            logits, target, reduction='none'
        ).mean(dim=1)
        
        if self.class_weights is not None:
            w = self.class_weights.to(device)[labels] 
            loss = (loss_per_sample * w).mean()
        else:
            loss = loss_per_sample.mean()
            
        return (loss, outputs) if return_outputs else loss

def compute_metrics_coral(eval_pred):
    logits, labels = eval_pred
    preds_binary = logits > 0
    pred_labels = preds_binary.sum(axis=1)
    qwk = cohen_kappa_score(labels, pred_labels, weights='quadratic')
    return {"qwk": qwk}

TÍNH TOÁN TRỌNG SỐ MẪU (SAMPLE-LEVEL WEIGHTS)...
Class weights (đã clip và chuẩn hóa):
[2.02 2.02 1.02 1.99 0.44 0.97 0.28 0.26 0.73 0.2  0.29 0.2  0.36 0.2
 0.58 1.35 2.02 2.02 2.02]


config.json:   0%|          | 0.00/753 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/654M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at UBC-NLP/ARBERTv2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
training_args = TrainingArguments(
    output_dir="../saved_models/arbert_lora_coral_sample_weight",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,               
    per_device_train_batch_size=8,    
    gradient_accumulation_steps=4,    
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="qwk",
    greater_is_better=True,
    bf16=True,   
    fp16=False,  
    seed=42
)

trainer = CORALTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["valid"],
    compute_metrics=compute_metrics_coral,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    class_weights=class_weights_tensor  
)

print("BẮT ĐẦU HUẤN LUYỆN (VỚI BFLOAT16 + CLIPPED SAMPLE WEIGHTS)...")
trainer.train()

trainer.save_model("../saved_models/arbert_lora_coral_best")
tokenizer.save_pretrained("../saved_models/arbert_lora_coral_best")

BẮT ĐẦU HUẤN LUYỆN (VỚI BFLOAT16 + CLIPPED SAMPLE WEIGHTS)...


  0%|          | 0/8535 [00:00<?, ?it/s]

{'loss': 0.0884, 'grad_norm': 1.9815771579742432, 'learning_rate': 0.00028242530755711773, 'epoch': 0.29}
{'loss': 0.073, 'grad_norm': 1.0747538805007935, 'learning_rate': 0.00026485061511423544, 'epoch': 0.59}
{'loss': 0.0682, 'grad_norm': 1.0334233045578003, 'learning_rate': 0.00024727592267135325, 'epoch': 0.88}


  0%|          | 0/457 [00:00<?, ?it/s]

{'eval_loss': 0.06972154229879379, 'eval_qwk': 0.7718509135541686, 'eval_runtime': 30.985, 'eval_samples_per_second': 235.921, 'eval_steps_per_second': 14.749, 'epoch': 1.0}
{'loss': 0.0612, 'grad_norm': 1.087946891784668, 'learning_rate': 0.000229701230228471, 'epoch': 1.17}
{'loss': 0.0578, 'grad_norm': 2.367386817932129, 'learning_rate': 0.00021212653778558875, 'epoch': 1.46}
{'loss': 0.0582, 'grad_norm': 1.2661744356155396, 'learning_rate': 0.00019455184534270648, 'epoch': 1.76}


...

  0%|          | 0/457 [00:00<?, ?it/s]

{'eval_loss': 0.06322114169597626, 'eval_qwk': 0.8124157585352819, 'eval_runtime': 30.8589, 'eval_samples_per_second': 236.884, 'eval_steps_per_second': 14.809, 'epoch': 4.0}
{'loss': 0.0405, 'grad_norm': 0.5560082197189331, 'learning_rate': 5.39543057996485e-05, 'epoch': 4.1}
{'loss': 0.0378, 'grad_norm': 1.4403877258300781, 'learning_rate': 3.6379613356766254e-05, 'epoch': 4.39}
{'loss': 0.0373, 'grad_norm': 0.8221835494041443, 'learning_rate': 1.8804920913884008e-05, 'epoch': 4.69}
{'loss': 0.0362, 'grad_norm': 1.2851113080978394, 'learning_rate': 1.2302284710017573e-06, 'epoch': 4.98}


  0%|          | 0/457 [00:00<?, ?it/s]

{'eval_loss': 0.06568580865859985, 'eval_qwk': 0.8193804359506444, 'eval_runtime': 31.0746, 'eval_samples_per_second': 235.24, 'eval_steps_per_second': 14.707, 'epoch': 5.0}
{'train_runtime': 3607.5892, 'train_samples_per_second': 75.71, 'train_steps_per_second': 2.366, 'train_loss': 0.052387567759258534, 'epoch': 5.0}


('../saved_models/arbert_lora_coral_best\\tokenizer_config.json',
 '../saved_models/arbert_lora_coral_best\\special_tokens_map.json',
 '../saved_models/arbert_lora_coral_best\\vocab.txt',
 '../saved_models/arbert_lora_coral_best\\added_tokens.json',
 '../saved_models/arbert_lora_coral_best\\tokenizer.json')

In [5]:
import numpy as np
from sklearn.metrics import cohen_kappa_score, classification_report

print("ĐANG LẤY LOGITS VÀ NHÃN TỪ TẬP VALIDATION...")
predictions_output = trainer.predict(tokenized_datasets["valid"])
logits = predictions_output.predictions  
true_labels = predictions_output.label_ids.astype(int)

print("ĐANG DÒ TÌM GLOBAL SHIFT (DELTA) TỐI ƯU...")

best_delta = 0.0
best_qwk = -1.0
qwk_history = []
deltas = np.arange(-2.0, 2.0, 0.01) 

for delta in deltas:
    preds_binary = logits > delta
    pred_labels = preds_binary.sum(axis=1)
    
    qwk = cohen_kappa_score(true_labels, pred_labels, weights='quadratic')
    
    if qwk > best_qwk:
        best_qwk = qwk
        best_delta = delta

print(f"Điểm dịch chuyển (Delta) tối ưu : {best_delta:.2f}")
print(f"QWK SAU KHI DỊCH CHUYỂN         : {best_qwk:.4f}")

final_preds_binary = logits > best_delta
final_pred_labels = final_preds_binary.sum(axis=1)

print("=== BÁO CÁO F1-SCORE VỚI DELTA TỐI ƯU ===")
target_names = [f"Level_{i+1}" for i in range(19)]
print(classification_report(true_labels, final_pred_labels, target_names=target_names, zero_division=0))

ĐANG LẤY LOGITS VÀ NHÃN TỪ TẬP VALIDATION...


  0%|          | 0/457 [00:00<?, ?it/s]

ĐANG DÒ TÌM GLOBAL SHIFT (DELTA) TỐI ƯU...
Điểm dịch chuyển (Delta) tối ưu : -0.46
QWK SAU KHI DỊCH CHUYỂN         : 0.8241
=== BÁO CÁO F1-SCORE VỚI DELTA TỐI ƯU ===
              precision    recall  f1-score   support

     Level_1       0.73      0.80      0.76        44
     Level_2       0.44      0.22      0.29        68
     Level_3       0.50      0.59      0.54       182
     Level_4       0.22      0.53      0.31        78
     Level_5       0.62      0.36      0.46       417
     Level_6       0.25      0.44      0.32       189
     Level_7       0.51      0.44      0.48       701
     Level_8       0.58      0.48      0.52       613
     Level_9       0.28      0.61      0.39       236
    Level_10       0.68      0.59      0.63      1012
    Level_11       0.19      0.25      0.21       409
    Level_12       0.46      0.41      0.44      1491
    Level_13       0.29      0.37      0.32       349
    Level_14       0.55      0.44      0.49      1072
    Level_15       0.23